### 1-1 Daum 뉴스기사 제목 스크래핑 하기

In [ ]:
import requests
import bs4
from bs4 import BeautifulSoup

url = 'https://news.daum.net/economy'
print(url)

req_header = {
    'referer':url,
    'user-agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

res = requests.get(url, headers = req_header)
res.encoding = "UTF-8"

print(type(res))
print(res.status_code)

if not res.ok:
    exit()

soup = BeautifulSoup(res.text, 'html.parser')
extracted = soup.select("ul.list_newsheadline2 li a")
print(type(extracted), len(extracted))

for ex in extracted:
    link = ex['href']
    title = ex.select_one("div.cont_thumb strong.tit_txt").text.strip()
    print(link)
    print(title)

#### 1-2 Daum 뉴스기사 제목 스크래핑 하기 코드를 섹션별로 처리하는 함수로 구현하기

In [ ]:
import requests
import bs4
from bs4 import BeautifulSoup

section_dict = {'기후/환경':'climate','사회':'society','경제':'economy','정치':'politics',\
             '국제':'world','문화':'culture','생활':'life','IT/과학':'tech','인물':'people'}

basic_url = 'https://news.daum.net/'

def print_news(section_name):
    url = basic_url + section_dict[section_name]

    req_header = {
        'referer':url,
        'user-agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
    }

    res = requests.get(url, headers = req_header)
    res.encoding = "UTF-8"

    if not res.ok:
        return

    print(f'======> {url} {section_name} 뉴스 <======')
    
    soup = BeautifulSoup(res.text, 'html.parser')
    extracted = soup.select("ul.list_newsheadline2 li a")

    for ex in extracted:
        link = ex['href']
        title = ex.select_one("div.cont_thumb strong.tit_txt").text.strip()
        print(link)
        print(title)
    

print_news('경제')
print_news('사회')


### 2-1 Nate 뉴스기사 제목 스크래핑하기

In [ ]:
import requests
import bs4
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import Image, display

url = "https://news.nate.com/recent?mid=n0100"
req_header = {
    'referer':url,
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

res = requests.get(url, headers = req_header)
if res.ok:
    soup = BeautifulSoup(res.text, 'html.parser')
    extracted = soup.select("a[href$='mid=n0100']") 

    for ex in extracted:
        if "recent" in ex['href']:
            continue;

        link = ex['href']
        title = ex.select_one("h2.tit").text.strip()
        src = ex.select_one("img")

        if src:
            display(Image(urljoin(url, src["src"])))

        print(title)
        print(urljoin(url, link))

        

### 2-2. 하나의 네이버 웹툰과 1개의 회차에 대한 Image 다운로드 하기

In [42]:
import requests
import bs4
from bs4 import BeautifulSoup
import os

def download_one_episode(title, no, url):

    req_header = {
        'referer':url,
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
    }

    res = requests.get(url, headers = req_header)

    if not res.ok:
        print(res.status_code)
        return

    soup = BeautifulSoup(res.text, 'html.parser')
    img_list = soup.select("img[src*='IMAG01']")

    file_path = os.path.join("img", title, str(no))
    os.makedirs(file_path, exist_ok=True)

    for index, img in enumerate(img_list):
        image = img['src']
        image_res = requests.get(image, headers = req_header)

        if not image_res.ok:
            print(image_res.status_code)
            return

        image_name = f"image{index+1}.jpg"
        image_path = os.path.join(file_path, image_name)

        with open(image_path, 'wb') as file:
            file.write(image_res.content)

download_one_episode('일렉시드',341,'https://comic.naver.com/webtoon/detail?titleId=717481&no=341&week=wed')
